In [3]:
# Import Required Libraries
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nsti-\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# Load Dataset

fake = pd.read_csv("Fake.csv")
real = pd.read_csv("True.csv")

In [5]:
fake.head(10)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017"
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017"
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017"
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017"
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017"


In [6]:
real.head(10)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"
5,"White House, Congress prepare for talks on spe...","WEST PALM BEACH, Fla./WASHINGTON (Reuters) - T...",politicsNews,"December 29, 2017"
6,"Trump says Russia probe will be fair, but time...","WEST PALM BEACH, Fla (Reuters) - President Don...",politicsNews,"December 29, 2017"
7,Factbox: Trump on Twitter (Dec 29) - Approval ...,The following statements were posted to the ve...,politicsNews,"December 29, 2017"
8,Trump on Twitter (Dec 28) - Global Warming,The following statements were posted to the ve...,politicsNews,"December 29, 2017"
9,Alabama official to certify Senator-elect Jone...,WASHINGTON (Reuters) - Alabama Secretary of St...,politicsNews,"December 28, 2017"


In [7]:
# Add labels

fake["label"] = 0
real["label"] = 1

In [8]:
# Combine datasets

data = pd.concat([fake, real])

In [9]:
# Shuffle data

data = data.sample(frac=1).reset_index(drop=True)

In [10]:
# Fixing column names (removing space)
data.columns = data.columns.str.strip()
data.columns

Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')

In [11]:
# Using ttile + text columns

data["content"] = data["title"] + " " + data["text"]

# Keep only needed columns
data = data[["content", "label"]]

# Rename column to match pipeline
data.rename(columns={"content": "text"}, inplace=True)

In [12]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

data["text"] = data["text"].apply(clean_text)
data


,text,label
0,obama says doubt civilians killed drone strike...,1
1,head senate panel says near deal russia iran s...,1
2,china media calls trump withdrawal paris accor...,1
3,whoa anderson cooper got bargained tried discr...,0
4,hacked commie george soros hacked dc leaks tha...,0
...,...,...
44893,donald trump shocked group women heard one fam...,0
44894,stranger fiction foundation vegas shooting sur...,0
44895,dinesh souza destroys leftist college student ...,0
44896,reince priebus says going forward trump comple...,0


In [13]:
# Tokenization

max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words = max_words)
tokenizer.fit_on_texts(data["text"])

X = tokenizer.texts_to_sequences(data["text"])
X = pad_sequences(X, maxlen = max_len)

y = data["label"]

In [14]:
# Save Tokenizer
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [15]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
    

In [16]:
# Build Model (LSTM)

model = Sequential()

model.add(Embedding(input_dim = max_words, output_dim = 64, input_length = max_len))
model.add(LSTM(64))
model.add(Dense(1, activation = 'sigmoid'))

model.compile(
    loss = 'binary_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

model.summary()

c:\Users\nsti-\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Train Model

model.fit(
    X_train,
    y_train,
    epochs = 3,
    batch_size = 64,
    validation_split = 0.1
)

Epoch 1/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - accuracy: 0.9066 - loss: 0.2767 - val_accuracy: 0.9574 - val_loss: 0.1322
Epoch 2/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.9783 - loss: 0.0737 - val_accuracy: 0.9813 - val_loss: 0.0631
Epoch 3/3
506/506 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - accuracy: 0.9890 - loss: 0.0382 - val_accuracy: 0.9836 - val_loss: 0.0531


In [18]:
# Evaluate the Model

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))


281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
Accuracy: 0.9864142538975501


In [19]:
# Testing the News 

def predict_news(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen = max_len)
    pred = model.predict(padded)[0][0]
    
    if pred > 0.5:
        print("REAL NEWS")
    else:
        print("FAKE NEWS")
        
   

In [22]:
# Test Function
def test_news(title, text):
    combined = title + " " + text
    predict_news(combined)

test_news('FBI Russia probe helped by Australian diplomat tip-off: NYT', 'WASHINGTON (Reuters) - Trump campaign adviser George Papadopoulos told an Australian diplomat in May 2016 that Russia had political dirt on Democratic presidential candidate Hillary Clinton, the New York Times reported on Saturday. The conversation between Papadopoulos and the diplomat, Alexander Downer, in London was a driving factor behind the FBIâ€™s decision to open a counter-intelligence investigation of Moscowâ€™s contacts with the Trump campaign, the Times reported. Two months after the meeting, Australian officials passed the information that came from Papadopoulos to their American counterparts when leaked Democratic emails began appearing online, according to the newspaper, which cited four current and former U.S. and foreign officials. Besides the information from the Australians, the probe by the Federal Bureau of Investigation was also propelled by intelligence from other friendly governments, including the British and Dutch, the Times said. Papadopoulos, a Chicago-based international energy lawyer, pleaded guilty on Oct. 30 to lying to FBI agents about contacts with people who claimed to have ties to top Russian officials. It was the first criminal charge alleging links between the Trump campaign and Russia. The White House has played down the former aideâ€™s campaign role, saying it was â€œextremely limitedâ€ and that any actions he took would have been on his own. The New York Times, however, reported that Papadopoulos helped set up a meeting between then-candidate Donald Trump and Egyptian President Abdel Fattah al-Sisi and edited the outline of Trumpâ€™s first major foreign policy speech in April 2016. The federal investigation, which is now being led by Special Counsel Robert Mueller, has hung over Trumpâ€™s White House since he took office almost a year ago. Some Trump allies have recently accused Muellerâ€™s team of being biased against the Republican president. Lawyers for Papadopoulos did not immediately respond to requests by Reuters for comment. Muellerâ€™s office declined to comment. Trumpâ€™s White House attorney, Ty Cobb, declined to comment on the New York Times report. â€œOut of respect for the special counsel and his process, we are not commenting on matters such as this,â€ he said in a statement. Mueller has charged four Trump associates, including Papadopoulos, in his investigation. Russia has denied interfering in the U.S. election and Trump has said there was no collusion between his campaign and Moscow. ')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
REAL NEWS


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
REAL NEWS
Prediction Score: <function predict_news at 0x0000011FFF705C60>


In [23]:
# Classifiacation Report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      4651
           1       0.99      0.98      0.99      4329

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

